# Advanced tutorial: Compare urban and forested watersheds

## Research question

**Are summer water temperatures different between predominantly developed and predominantly forested STREAM watersheds?**

## `streamwq` functions used

- `StreamClient.find_sites()`
- `StreamClient.read_land_cover()`
- `StreamClient.read_water_quality()`
- `filter_season()`
- `resample_timeseries()`

In [ ]:
from streamwq.client import StreamClient
from streamwq.temporal import filter_season, resample_timeseries

stream = StreamClient()

## 1. Read land cover for stations with temperature observations

In [ ]:
sites = stream.find_sites(variables="WTemp_C")
site_ids = sites["STREAM_ID"].tolist()
developed_columns = [
    "Developed Open Space", "Developed Low Intensity",
    "Developed Medium Intensity", "Developed High Intensity",
]
forest_columns = ["Deciduous Forest", "Evergreen Forest", "Mixed Forest"]
land_cover = stream.read_land_cover(
    site_ids, variables=developed_columns + forest_columns
)

## 2. Define comparison groups

These 50% thresholds are transparent tutorial choices, not universal scientific definitions. The most recent land-cover record is used for each station.

In [ ]:
latest = (
    land_cover.sort_values("DateTime")
    .groupby("STREAM_ID", as_index=False)
    .tail(1)
    .copy()
)
latest["developed_fraction"] = latest[developed_columns].sum(axis=1)
latest["forest_fraction"] = latest[forest_columns].sum(axis=1)
land_cover_scale = 1.0 if latest[developed_columns + forest_columns].max().max() <= 1 else 100.0
majority_threshold = 0.50 * land_cover_scale
latest["watershed_group"] = "other"
latest.loc[latest["developed_fraction"] > majority_threshold, "watershed_group"] = "developed"
latest.loc[latest["forest_fraction"] > majority_threshold, "watershed_group"] = "forested"
latest["watershed_group"].value_counts()

## 3. Calculate station-level summer temperature

In [ ]:
comparison_ids = latest.loc[
    latest["watershed_group"].isin(["developed", "forested"]), "STREAM_ID"
].tolist()
temperature = stream.read_water_quality(
    comparison_ids, "WTemp_C", quality="approved"
)
summer = filter_season(temperature, "summer")
daily_summer = resample_timeseries(summer, "WTemp_C", "daily", "mean")
station_means = daily_summer.groupby("STREAM_ID", as_index=False)["WTemp_C"].mean()
analysis = station_means.merge(
    latest[["STREAM_ID", "watershed_group"]], on="STREAM_ID"
)
analysis.groupby("watershed_group")["WTemp_C"].describe()